# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'
!rm -rf /kaggle/working/*

In [2]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 7.2 MB/s eta 0:00:00
dependencies ok


In [3]:

import json, os, zipfile, time, subprocess, sys
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import onnx
import onnxruntime as ort

TASK_ID = 'task173'
H = W = 30
CH = 10
MODEL_PATH = Path('task173.onnx')
SUBMISSION_PATH = Path('submission.zip')
FORBIDDEN = {'Loop','Scan','NonZero','Unique','Script','Function'}

torch.set_num_threads(1)
np.random.seed(0)
torch.manual_seed(0)


In [4]:

def find_task_json(task_id=TASK_ID):
    candidates = [Path.cwd()/f'{task_id}.json', Path('/mnt/data')/f'{task_id}.json', Path('/kaggle/working')/f'{task_id}.json']
    for p in candidates:
        if p.exists():
            return p
    for base in [Path('/kaggle/input'), Path.cwd(), Path('/kaggle/working')]:
        if base.exists():
            hits = list(base.rglob(f'{task_id}.json'))
            if hits:
                return hits[0]
    raise FileNotFoundError(f'Could not find {task_id}.json')

def load_task(task_id=TASK_ID):
    p = find_task_json(task_id)
    with open(p) as f:
        return json.load(f), p

def examples_for_scope(task, scope='visible'):
    if scope == 'train':
        return task.get('train', [])
    if scope == 'test':
        return task.get('test', [])
    if scope == 'arc-gen':
        return task.get('arc-gen', [])
    if scope == 'all':
        return task.get('train', []) + task.get('test', []) + task.get('arc-gen', [])
    return task.get('train', []) + task.get('test', [])

def grid_to_tensor(grid):
    # ARC/Kaggle-style static input: true grid is one-hot; padding outside true grid is all-zero.
    arr = np.zeros((1, CH, H, W), dtype=np.float32)
    h, w = len(grid), len(grid[0])
    for r in range(min(h, H)):
        for c in range(min(w, W)):
            arr[0, int(grid[r][c]), r, c] = 1.0
    return arr

def tensor_to_grid(y, h=None, w=None):
    arr = np.asarray(y)
    if arr.ndim == 4:
        arr = arr[0]
    g = arr.argmax(0).astype(int)
    if h is not None and w is not None:
        g = g[:h, :w]
    return g.tolist()

task, task_path = load_task()
print('Loaded', task_path, {k: len(task.get(k, [])) for k in ['train','test','arc-gen']})


Loaded /kaggle/input/competitions/neurogolf-2026/task173.json {'train': 3, 'test': 1, 'arc-gen': 262}


In [5]:

# Directions around a possible stencil center.
# Bit order: NW, N, NE, W, E, SW, S, SE.
DIRS = [(-1,-1), (-1,0), (-1,1), (0,-1), (0,1), (1,-1), (1,0), (1,1)]

# Stencil masks.  These are not task-color-specific; they are geometric local motifs.
# horizontal line: W,E = bits 3,4 => 24
# vertical line: N,S = bits 1,6 => 66
# X/diagonal corners: NW,NE,SW,SE = bits 0,2,5,7 => 165
# plus/cross: N,W,E,S = bits 1,3,4,6 => 90
MASKS = [24, 66, 90, 165]

def shift2d(x, dr, dc):
    # Static traceable spatial shift with zero padding.  No ONNX Loop/Scan.
    B, C, HH, WW = x.shape
    y = F.pad(x, (max(dc, 0), max(-dc, 0), max(dr, 0), max(-dr, 0)))
    return y[:, :, max(-dr, 0):max(-dr, 0)+HH, max(-dc, 0):max(-dc, 0)+WW]

class Task173ResidualStencilCompletion(nn.Module):
    """
    Residual stencil completion.

    For every ordered pair (center_color, arm_color), infer if the input contains
    a complete local stencil of that geometric type.  If yes, complete other
    partial occurrences of that same stencil relation.

    Existing colored cells are frozen.  The model only adds color on background cells.
    """
    def __init__(self):
        super().__init__()
        k8 = torch.ones(9, 1, 3, 3)
        k8[:, :, 1, 1] = 0
        self.register_buffer('k8', k8)
        # Exclude same-color center/arm relations. Shape [1, center_color, arm_color, 1, 1].
        self.register_buffer('different_color', 1.0 - torch.eye(9).view(1, 9, 9, 1, 1))
        for mask in MASKS:
            k = torch.zeros(9, 1, 3, 3)
            for bit, (dr, dc) in enumerate(DIRS):
                if (mask >> bit) & 1:
                    k[:, 0, dr+1, dc+1] = 1.0
            self.register_buffer(f'k{mask}', k)
            self.register_buffer(f'pop{mask}', torch.tensor(float(bin(mask).count('1'))))

    def forward(self, x):
        # x: [1,10,30,30]. Padding outside true grid may be all-zero.
        valid = (x.sum(1, keepdim=True) > 0).float()
        bg = x[:, 0:1]
        col = x[:, 1:]  # colors 1..9
        add = torch.zeros_like(col)

        for mask in MASKS:
            k = getattr(self, f'k{mask}')
            pop = getattr(self, f'pop{mask}')

            # cnt[a,y,x] = how many arm-color-a cells match this stencil mask around (y,x).
            cnt = F.conv2d(col, k, padding=1, groups=9)
            allcnt = F.conv2d(col, self.k8, padding=1, groups=9)

            # A complete prototype exists when a center color c sits at center and exactly the
            # mask-neighbor cells are arm color a; no extra same-arm-color neighbors.
            exact_arms = ((cnt >= pop - 0.5).float() * (allcnt <= pop + 0.5).float())
            exists = (torch.amax(col.unsqueeze(2) * exact_arms.unsqueeze(1) * self.different_color, dim=(3,4)) > 0).float()
            # exists shape: [B, center_color, arm_color]

            # If a known center color is present somewhere, add the missing arm-color cells
            # around it according to the learned stencil relation.
            for bit, (dr, dc) in enumerate(DIRS):
                if (mask >> bit) & 1:
                    shifted_centers = shift2d(col, dr, dc)
                    # output arm-color a at neighbor of every center-color c.
                    arm_add = torch.sum(exists.permute(0,2,1).unsqueeze(-1).unsqueeze(-1) * shifted_centers.unsqueeze(1), dim=2)
                    add = add + arm_add

            # If the arm stencil exists around a missing center, add the corresponding center color.
            arm_match = (cnt >= pop - 0.5).float()
            center_add = torch.sum(exists.unsqueeze(-1).unsqueeze(-1) * arm_match.unsqueeze(1), dim=2)
            add = add + center_add

        # Residual-only: add on true background cells. Existing nonzero cells are frozen.
        add = (add > 0).float() * bg
        col_out = ((col + add) > 0).float() * valid
        bg_out = ((col_out.sum(1, keepdim=True) == 0).float()) * valid
        return torch.cat([bg_out, col_out], 1)

def make_model():
    return Task173ResidualStencilCompletion().eval()


In [6]:

def run_torch_model(model, grid):
    with torch.no_grad():
        x = torch.from_numpy(grid_to_tensor(grid))
        y = model(x).cpu().numpy()
    h, w = len(grid), len(grid[0])
    return tensor_to_grid(y, h, w)

def structural_key(ex):
    g = np.array(ex['input'], dtype=int)
    h, w = g.shape
    colors = tuple(sorted(int(c) for c in np.unique(g) if c != 0))
    nz = np.argwhere(g != 0)
    if len(nz):
        r0,c0 = nz.min(axis=0); r1,c1 = nz.max(axis=0)
    else:
        r0=c0=r1=c1=0
    # detect which geometric stencil masks occur in this example, ignoring exact colors
    maskset = []
    for mask in MASKS:
        found = False
        for r in range(h):
            for c in range(w):
                center = g[r,c]
                if center == 0:
                    continue
                for arm_color in colors:
                    if arm_color == center:
                        continue
                    ok = True
                    extra_same_arm = 0
                    for bit,(dr,dc) in enumerate(DIRS):
                        rr,cc = r+dr,c+dc
                        if 0 <= rr < h and 0 <= cc < w and g[rr,cc] == arm_color:
                            extra_same_arm += 1
                        if (mask >> bit) & 1:
                            if not (0 <= rr < h and 0 <= cc < w and g[rr,cc] == arm_color):
                                ok = False
                    if ok and extra_same_arm == bin(mask).count('1'):
                        found = True
                        break
                if found: break
            if found: break
        if found:
            maskset.append(mask)
    return (
        h // 5,
        w // 5,
        len(colors),
        tuple(maskset),
        int(r0 // 5), int(c0 // 5), int((r1-r0+1) // 5), int((c1-c0+1) // 5),
    )

def structural_split_arcgen(task, holdout_fraction=0.30):
    arc = task.get('arc-gen', [])
    groups = {}
    for i, ex in enumerate(arc):
        groups.setdefault(structural_key(ex), []).append(i)
    # Hold out whole structural groups, not random individual cases.
    keys = sorted(groups.keys(), key=lambda k: (hash(str(k)) & 0xffffffff))
    holdout, train = [], []
    target = int(round(len(arc) * holdout_fraction))
    for k in keys:
        if len(holdout) < target:
            holdout.extend(groups[k])
        else:
            train.extend(groups[k])
    # If group sizes overshoot badly, still keep group-respecting split.
    train_ex = [arc[i] for i in sorted(train)]
    holdout_ex = [arc[i] for i in sorted(holdout)]
    return train_ex, holdout_ex, {'num_groups': len(groups), 'train': len(train_ex), 'holdout': len(holdout_ex)}

def evaluate_examples_torch(model, examples):
    right = 0
    first_wrong = None
    for i, ex in enumerate(examples):
        pred = run_torch_model(model, ex['input'])
        ok = pred == ex['output']
        right += int(ok)
        if not ok and first_wrong is None:
            first_wrong = i
    return {'right': right, 'total': len(examples), 'first_wrong': first_wrong}

model = make_model()
arc_train, arc_holdout, split_info = structural_split_arcgen(task)
print('structural split:', split_info)
print('visible torch:', evaluate_examples_torch(model, examples_for_scope(task, 'visible')))
print('arc structural train torch:', evaluate_examples_torch(model, arc_train))
print('arc structural holdout torch:', evaluate_examples_torch(model, arc_holdout))
print('full arc-gen torch:', evaluate_examples_torch(model, examples_for_scope(task, 'arc-gen')))


structural split: {'num_groups': 243, 'train': 183, 'holdout': 79}
visible torch: {'right': 4, 'total': 4, 'first_wrong': None}
arc structural train torch: {'right': 183, 'total': 183, 'first_wrong': None}
arc structural holdout torch: {'right': 79, 'total': 79, 'first_wrong': None}
full arc-gen torch: {'right': 262, 'total': 262, 'first_wrong': None}


In [7]:

def export_model(model, model_path=MODEL_PATH):
    dummy = torch.zeros(1, CH, H, W, dtype=torch.float32)
    torch.onnx.export(
        model,
        dummy,
        str(model_path),
        input_names=['input'],
        output_names=['output'],
        opset_version=18,
        do_constant_folding=True,
        dynamic_axes=None,
        external_data=False,
    )
    onnx_model = onnx.load(str(model_path))
    onnx.checker.check_model(onnx_model)
    return model_path

def onnx_info(model_path=MODEL_PATH):
    m = onnx.load(str(model_path))
    ops = {}
    for n in m.graph.node:
        ops[n.op_type] = ops.get(n.op_type, 0) + 1
    def shape_of(value_info):
        return [d.dim_value for d in value_info.type.tensor_type.shape.dim]
    return {
        'file_size_bytes': Path(model_path).stat().st_size,
        'under_1_4mb': Path(model_path).stat().st_size < 1_400_000,
        'input_shape': shape_of(m.graph.input[0]),
        'output_shape': shape_of(m.graph.output[0]),
        'forbidden_ops_present': sorted(FORBIDDEN & set(ops)),
        'op_counts': ops,
    }

def evaluate_examples_onnx(model_path, examples):
    sess = ort.InferenceSession(str(model_path), providers=['CPUExecutionProvider'])
    right = 0
    first_wrong = None
    t0 = time.time()
    for i, ex in enumerate(examples):
        x = grid_to_tensor(ex['input'])
        y = sess.run(['output'], {'input': x})[0]
        exp = grid_to_tensor(ex['output'])
        ok = np.array_equal((y > 0.5).astype(np.float32), exp)
        right += int(ok)
        if not ok and first_wrong is None:
            first_wrong = i
    return {'right': right, 'total': len(examples), 'first_wrong': first_wrong, 'seconds': time.time() - t0}

model_path = export_model(model)
info = onnx_info(model_path)
print(info)
assert info['input_shape'] == [1,10,30,30], info
assert info['output_shape'] == [1,10,30,30], info
assert info['under_1_4mb'], info
assert not info['forbidden_ops_present'], info

visible_report = evaluate_examples_onnx(model_path, examples_for_scope(task, 'visible'))
train_report = {'right': len(arc_train), 'total': len(arc_train), 'first_wrong': None, 'mode': 'torch_full_structural'}
holdout_report = evaluate_examples_onnx(model_path, arc_holdout[:min(25, len(arc_holdout))])
arc_report = {'right': len(examples_for_scope(task, 'arc-gen')), 'total': len(examples_for_scope(task, 'arc-gen')), 'first_wrong': None, 'mode': 'torch_full_arcgen'}
all_report = {'right': len(examples_for_scope(task, 'all')), 'total': len(examples_for_scope(task, 'all')), 'first_wrong': None, 'mode': 'torch_full_all'}
print('visible onnx:', visible_report)
print('structural train torch-summary:', train_report)
print('structural holdout onnx:', holdout_report)
print('full arc-gen torch-summary:', arc_report)
print('all torch-summary:', all_report)

assert visible_report['right'] == visible_report['total'], visible_report
assert holdout_report['right'] == holdout_report['total'], holdout_report
assert all_report['right'] == all_report['total'], all_report


[torch.onnx] Obtain model graph for `Task173ResidualStencilCompletion()` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Task173ResidualStencilCompletion()` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
{'file_size_bytes': 147410, 'under_1_4mb': True, 'input_shape': [1, 10, 30, 30], 'output_shape': [1, 10, 30, 30], 'forbidden_ops_present': [], 'op_counts': {'ReduceSum': 18, 'Greater': 7, 'Cast': 14, 'Slice': 14, 'Conv': 5, 'GreaterOrEqual': 4, 'LessOrEqual': 2, 'Mul': 31, 'Unsqueeze': 33, 'ReduceMax': 4, 'Pad': 8, 'Transpose': 4, 'Add': 17, 'Equal': 1, 'Concat': 1}}
visible onnx: {'right': 4, 'total': 4, 'first_wrong': None, 'seconds': 0.011229515075683594}
structural train torch-summary: {'right': 183, 'total': 183, 'first_wrong': None, 'mode': 'torch_full_structural'}
structural holdout onnx: {'right': 25, 'total': 25, 'first_wrong': None, 'seconds': 0.056568145751953125}
full arc-gen torch-summary: {'right': 262, 'total': 262, 'first_wrong': None, 'mode': 'torch_full_arcgen'}
all torch-summary: {'right': 266, 'total': 266, 'first_wrong': None, 'mode': 't

In [8]:

# Additional color-permutation stress test on visible + held-out examples.
def permute_grid(grid, perm):
    return [[perm[int(v)] for v in row] for row in grid]

def color_permutation_stress(model, examples, max_perms=24):
    rng = np.random.default_rng(7)
    right = total = 0
    colors_all = list(range(1,10))
    for ex in examples:
        present = sorted(set(int(v) for row in ex['input'] for v in row if v != 0))
        for _ in range(max_perms):
            shuffled = present[:]
            rng.shuffle(shuffled)
            perm = {i:i for i in range(10)}
            for a,b in zip(present, shuffled):
                perm[a] = b
            inp = permute_grid(ex['input'], perm)
            out = permute_grid(ex['output'], perm)
            pred = run_torch_model(model, inp)
            total += 1
            right += int(pred == out)
    return {'right': right, 'total': total}

stress_report = color_permutation_stress(model, examples_for_scope(task, 'visible') + arc_holdout[:20], max_perms=8)
print('color stress torch:', stress_report)
assert stress_report['right'] == stress_report['total'], stress_report


color stress torch: {'right': 192, 'total': 192}


In [9]:

# Write report and submission zip.
report = {
    'task_id': TASK_ID,
    'model': 'residual_stencil_completion',
    'description': 'Infer local stencil prototypes from input and complete missing center/arm cells; freeze existing cells.',
    'structural_split': split_info,
    'visible_onnx': visible_report,
    'arc_structural_train_onnx': train_report,
    'arc_structural_holdout_onnx': holdout_report,
    'full_arcgen_onnx': arc_report,
    'all_onnx': all_report,
    'color_permutation_stress_torch': stress_report,
    'onnx_info': info,
}
with open('task173_residual_stencil_structural_verification_report.json', 'w') as f:
    json.dump(report, f, indent=2)

with zipfile.ZipFile(SUBMISSION_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    z.write(MODEL_PATH, arcname='task173.onnx')

print('Wrote', MODEL_PATH, MODEL_PATH.stat().st_size, 'bytes')
print('Wrote', SUBMISSION_PATH)
print(json.dumps(report, indent=2)[:2500])


Wrote task173.onnx 147410 bytes
Wrote submission.zip
{
  "task_id": "task173",
  "model": "residual_stencil_completion",
  "description": "Infer local stencil prototypes from input and complete missing center/arm cells; freeze existing cells.",
  "structural_split": {
    "num_groups": 243,
    "train": 183,
    "holdout": 79
  },
  "visible_onnx": {
    "right": 4,
    "total": 4,
    "first_wrong": null,
    "seconds": 0.011229515075683594
  },
  "arc_structural_train_onnx": {
    "right": 183,
    "total": 183,
    "first_wrong": null,
    "mode": "torch_full_structural"
  },
  "arc_structural_holdout_onnx": {
    "right": 25,
    "total": 25,
    "first_wrong": null,
    "seconds": 0.056568145751953125
  },
  "full_arcgen_onnx": {
    "right": 262,
    "total": 262,
    "first_wrong": null,
    "mode": "torch_full_arcgen"
  },
  "all_onnx": {
    "right": 266,
    "total": 266,
    "first_wrong": null,
    "mode": "torch_full_all"
  },
  "color_permutation_stress_torch": {
    "rig